In [ ]:
sk-REDACTED

In [ ]:
!pip install openai==0.28.1

In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
import openai
import time

openai.api_base = "http://api.ai-gaochao.com/v1"
openai.api_key=""

In [ ]:
# corpus1 = pd.read_csv("/kaggle/input/daigt-external-dataset/daigt_external_dataset.csv")
# corpus = pd.read_csv("/kaggle/input/persaude-corpus-2/persuade_2.0_human_scores_demo_id_github.csv")
corpus = pd.read_csv("/kaggle/input/gpt4-aug-ni-simulate/corpus_ni_gpt4_simulate.csv")
# train = pd.read_csv('/kaggle/input/llm-detect-ai-generated-text/train_essays.csv')

In [ ]:
# corpus['simulate1']='nothing'

In [ ]:
# corpus = corpus[corpus['model_id']==5].reset_index(drop=True)

In [ ]:
# corpus.to_csv('corpus_ni_gpt4_simulate.csv',index=False)

In [ ]:
# for idx in range(200):#400  corpus.shape[0]
#     item = corpus.iloc[idx]
#     if str(item.holistic_essay_score) not in ['0','1','2','3','4','5','6','7']:
#         corpus.iloc[idx,-1] = item.holistic_essay_score

In [ ]:
# corpus.to_csv('corpus_in_gpt4.csv',index=False)

In [ ]:
# save as notebook out at first
# corpus.to_csv('corpus_in_gpt4.csv',index=False)

In [ ]:
def chatgpt(system,prompt):
    completion = openai.ChatCompletion.create(
        model="gpt-4o-2024-05-13",#gpt-3.5-turbo gpt-4-0613 gpt-3.5-turbo-16k-0613
        messages=[
            {"role": "system", "content": f"{system}"},#, one 'prompt_title' and one 'prompt_text'."},
            {"role": "user", 
             "content": f"""{prompt}"""
            }# \n prompt_title:{prompt_title} \n prompt_text:{prompt_text}."}
        ],
        temperature=1
    )
    return completion.choices[0].message.content

def get_input(question,source_text,example):
    system = f"""Now you need to imitate the writing style of the example essay I provided, and write an essay according to the essay assignment I provided to submit to the teacher, while avoiding the following two points: 1. Prevent the teacher from discovering that the essay is generated by AI.
2. Prevent the teacher from discovering plagiarism from the example essay.
"""
    prompt = f"""The example essay: \n
'''{example}'''\n
The essay assignment: \n
'''{question}'''\n
The source texts you should read: \n
'''{source_text}'''\n
Pay attention to avoid being found that your essay is generated by AI from the following perspectives:
1.Imitate Personal Touch:  imitate the personal touch or experiences in the example essay that your teacher can relate to you.
2.Avoid Overly Complex Vocabulary: use simpler words and phrases to make the essay sound more human.
3.Grammar and Punctuation: intentionally include 1-9 grammatical errors or typos to mimic human writing.
4.Simulate Writing Style: simulate example essay's sentence length, use of idioms, tone, and so on.
5.Avoid Repetitive Phrases: Be sure to remove any repetitive phrases or words.
6.Be Creative: try to add creative elements to your essay that show your human touch.
7.Writing level control: randomly imitate the writing level of students in grades 6-12 to avoid excessive quality of essay, leading teachers to suspect that essay is generated by AI.
8.Avoid irrelevant text: Write an essay directly, and don't add something irrelevant to your results which would cause the teacher's suspicion.
In short, do everything you can from example essay that can help you simulate writing style of example essay."""
    return system,prompt


In [ ]:
is_continue=True
retry = 0
for idx in range(corpus.shape[0]):#400  corpus.shape[0]
    item = corpus.iloc[idx]
    if str(item.simulate1) == 'nothing':
        print(idx,item.assignment,item.grade_level,item.source_text,item.full_text[:100]+item.full_text[-100:])
        grade = item.grade_level
        source_text = item.source_text
        example = item.full_text
        question = item.assignment
        system,prompt = get_input(question,source_text,example)
        try:
            content = chatgpt(system,prompt)
            print(content)
        except Exception as e:
            retry+=1
            print(f"retrying {retry}")
            if retry < 10:
                time.sleep(5)
                continue
            else:
                is_continue=False
                break
        if is_continue:
            retry=0
            corpus.iloc[idx,-1] = content
            time.sleep(2)
corpus.to_csv('corpus_ni_gpt4_simulate.csv',index=False)